# Проект - Предсказание цен на жилую недвижимость

## Цель проекта:
- обучить модель линейной регрессии на данных о жилье в Калифорнии в 1990 году

## План проекта:
- 1. **Настройка окружения** - инициализация Spark сессии
- 2. **Загрузка и первичный анализ данных**
- 3. **Предобработка данных** - обработка пропусков, кодирование категорий
- 4. **Построение моделей** - две версии линейной регрессии
- 5. **Оценка качества** - сравнение метрик RMSE, MAE, R2

**Инициализация Spark сессии**

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.ml.feature import VectorAssembler


spark = SparkSession.builder \
    .appName("California Housing Linear Regression") \
    .getOrCreate()

print("Spark сессия создана")
print(f"Spark версия: {spark.version}")

Spark сессия создана
Spark версия: 3.0.2


**Загрузка данных**

In [4]:
file_path = "/datasets/housing.csv"
housing_df = spark.read.csv(file_path, header=True, inferSchema=True)

print("Данные загружены:")
print(f"Количество строк: {housing_df.count()}")
print(f"Количество колонок: {len(housing_df.columns)}")

print("\n Структура данных:")
housing_df.printSchema()

Данные загружены:
Количество строк: 20640
Количество колонок: 10

 Структура данных:
root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- median_house_value: double (nullable = true)
 |-- ocean_proximity: string (nullable = true)



**Первичный анализ данных**

In [6]:
print("Первые 5 строк данных:")
housing_df.show(5)

print("\n Проверка пропусков:")
from pyspark.sql.functions import col, sum

null_counts = housing_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in housing_df.columns])
print("Количество пропусков по колонкам:")
null_counts.show()

Первые 5 строк данных:
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|  -122.23|   37.88|              41.0|      880.0|         129.0|     322.0|     126.0|       8.3252|          452600.0|       NEAR BAY|
|  -122.22|   37.86|              21.0|     7099.0|        1106.0|    2401.0|    1138.0|       8.3014|          358500.0|       NEAR BAY|
|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|
|  -122.25|   37.85|              52.0|     1274.0|         235.0|     558.0|     219.0|       5.6431|          341300.0|       NEAR BAY|
|  -122.25|

**Анализ категориальной переменной**

In [7]:
print("Распределение по близости к океану:")
housing_df.groupBy("ocean_proximity").count().orderBy("count", ascending=False).show()

print("\n Статистика по числовым колонкам:")
numeric_cols = [c for c in housing_df.columns if c != 'ocean_proximity']
housing_df.select(numeric_cols).describe().show()

Распределение по близости к океану:


+---------------+-----+
|ocean_proximity|count|
+---------------+-----+
|      <1H OCEAN| 9136|
|         INLAND| 6551|
|     NEAR OCEAN| 2658|
|       NEAR BAY| 2290|
|         ISLAND|    5|
+---------------+-----+


 Статистика по числовым колонкам:
+-------+-------------------+-----------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+
|summary|          longitude|         latitude|housing_median_age|       total_rooms|    total_bedrooms|        population|       households|     median_income|median_house_value|
+-------+-------------------+-----------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+
|  count|              20640|            20640|             20640|             20640|             20433|             20640|            20640|             20640|             20640|
|   mean|-119.56970445736148

**Обработка пропусков**

In [8]:
median_bedrooms = housing_df.approxQuantile("total_bedrooms", [0.5], 0.01)[0]
print(f"Медиана total_bedrooms: {median_bedrooms}")

from pyspark.sql.functions import when

housing_df_filled = housing_df.fillna({"total_bedrooms": median_bedrooms})

print("Проверка заполнения пропусков:")
null_check = housing_df_filled.select([sum(col(c).isNull().cast("int")).alias(c) for c in housing_df_filled.columns])
null_check.show()

Медиана total_bedrooms: 433.0
Проверка заполнения пропусков:
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|        0|       0|                 0|          0|             0|         0|         0|            0|                 0|              0|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+



**Подготовка данных для машинного обучения**

In [9]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline

print("Подготовка категориальной переменной...")

indexer = StringIndexer(inputCol="ocean_proximity", outputCol="ocean_proximity_index")

encoder = OneHotEncoder(inputCol="ocean_proximity_index", outputCol="ocean_proximity_encoded")

categorical_pipeline = Pipeline(stages=[indexer, encoder])

categorical_model = categorical_pipeline.fit(housing_df_filled)
housing_df_encoded = categorical_model.transform(housing_df_filled)

print("One-Hot Encoding применен:")
housing_df_encoded.select("ocean_proximity", "ocean_proximity_index", "ocean_proximity_encoded").show(5)

Подготовка категориальной переменной...


One-Hot Encoding применен:
+---------------+---------------------+-----------------------+
|ocean_proximity|ocean_proximity_index|ocean_proximity_encoded|
+---------------+---------------------+-----------------------+
|       NEAR BAY|                  3.0|          (4,[3],[1.0])|
|       NEAR BAY|                  3.0|          (4,[3],[1.0])|
|       NEAR BAY|                  3.0|          (4,[3],[1.0])|
|       NEAR BAY|                  3.0|          (4,[3],[1.0])|
|       NEAR BAY|                  3.0|          (4,[3],[1.0])|
+---------------+---------------------+-----------------------+
only showing top 5 rows



**Подготовка фич для моделей**

In [11]:
numeric_features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 
                   'total_bedrooms', 'population', 'households', 'median_income']


assembler_all = VectorAssembler(
    inputCols=numeric_features + ['ocean_proximity_encoded'],
    outputCol="features_all"
)

assembler_numeric = VectorAssembler(
    inputCols=numeric_features,
    outputCol="features_numeric"
)

housing_df_prepared = assembler_all.transform(housing_df_encoded)
housing_df_prepared = assembler_numeric.transform(housing_df_prepared)

print("Фичи подготовлены. Структура данных:")
housing_df_prepared.select("features_all", "features_numeric", "median_house_value").show(5, truncate=False)

✅ Фичи подготовлены. Структура данных:
+-----------------------------------------------------------------------+-------------------------------------------------------+------------------+
|features_all                                                           |features_numeric                                       |median_house_value|
+-----------------------------------------------------------------------+-------------------------------------------------------+------------------+
|[-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,0.0,0.0,0.0,1.0]    |[-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252]    |452600.0          |
|[-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,0.0,0.0,0.0,1.0]|[-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014]|358500.0          |
|[-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,0.0,0.0,0.0,1.0]   |[-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574]   |352100.0          |
|[-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,0.0,0.0,0.0,1.

**Разделение данных на train/test**

In [12]:
train_df, test_df = housing_df_prepared.randomSplit([0.8, 0.2], seed=42)

print("Разделение данных:")
print(f"Обучающая выборка: {train_df.count()} записей")
print(f"Тестовая выборка: {test_df.count()} записей")

print("\n Распределение целевой переменной:")
train_df.select("median_house_value").describe().show()
test_df.select("median_house_value").describe().show()

Разделение данных:


Обучающая выборка: 16560 записей


Тестовая выборка: 4080 записей

 Распределение целевой переменной:
+-------+------------------+
|summary|median_house_value|
+-------+------------------+
|  count|             16560|
|   mean|205596.39806763286|
| stddev|114804.48784864499|
|    min|           14999.0|
|    max|          500001.0|
+-------+------------------+

+-------+------------------+
|summary|median_house_value|
+-------+------------------+
|  count|              4080|
|   mean|211967.57573529411|
| stddev|117640.36027224957|
|    min|           14999.0|
|    max|          500001.0|
+-------+------------------+



**Построение первой модели (все фичи)**

In [13]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

print("Обучаем модель 1 (все фичи)...")

lr_all = LinearRegression(
    featuresCol="features_all",
    labelCol="median_house_value",
    predictionCol="prediction_all"
)

lr_model_all = lr_all.fit(train_df)

predictions_all = lr_model_all.transform(test_df)

print("Модель 1 обучена!")
print(f"Коэффициент детерминации (R²): {lr_model_all.summary.r2:.4f}")

Обучаем модель 1 (все фичи)...


25/11/24 18:26:09 WARN Instrumentation: [02b1e795] regParam is zero, which might cause numerical instability and overfitting.
25/11/24 18:26:09 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
25/11/24 18:26:09 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS
25/11/24 18:26:10 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeSystemLAPACK
25/11/24 18:26:10 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeRefLAPACK


Модель 1 обучена!
Коэффициент детерминации (R²): 0.6463


**Построение второй модели (только числовые фичи)**

In [14]:
print("Обучаем модель 2 (только числовые фичи)...")

lr_numeric = LinearRegression(
    featuresCol="features_numeric", 
    labelCol="median_house_value",
    predictionCol="prediction_numeric"
)

lr_model_numeric = lr_numeric.fit(train_df)

predictions_numeric = lr_model_numeric.transform(test_df)

print("Модель 2 обучена!")
print(f"Коэффициент детерминации (R²): {lr_model_numeric.summary.r2:.4f}")

Обучаем модель 2 (только числовые фичи)...


25/11/24 18:27:26 WARN Instrumentation: [246a7304] regParam is zero, which might cause numerical instability and overfitting.


Модель 2 обучена!
Коэффициент детерминации (R²): 0.6366


**Оценка моделей по всем метрикам**

In [15]:
from pyspark.ml.evaluation import RegressionEvaluator

print("ОЦЕНКА КАЧЕСТВА МОДЕЛЕЙ НА ТЕСТОВОЙ ВЫБОРКЕ")

evaluator_rmse = RegressionEvaluator(labelCol="median_house_value", predictionCol="prediction_all", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="median_house_value", predictionCol="prediction_all", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="median_house_value", predictionCol="prediction_all", metricName="r2")

rmse_all = evaluator_rmse.evaluate(predictions_all)
mae_all = evaluator_mae.evaluate(predictions_all)
r2_all = evaluator_r2.evaluate(predictions_all)

print("\n МОДЕЛЬ 1 (ВСЕ ФИЧИ):")
print(f"   RMSE: {rmse_all:.2f}")
print(f"   MAE:  {mae_all:.2f}") 
print(f"   R²:   {r2_all:.4f}")

ОЦЕНКА КАЧЕСТВА МОДЕЛЕЙ НА ТЕСТОВОЙ ВЫБОРКЕ

 МОДЕЛЬ 1 (ВСЕ ФИЧИ):
   RMSE: 70786.68
   MAE:  50863.76
   R²:   0.6378


**Оценка второй модели**

In [17]:
evaluator_rmse_numeric = RegressionEvaluator(labelCol="median_house_value", predictionCol="prediction_numeric", metricName="rmse")
evaluator_mae_numeric = RegressionEvaluator(labelCol="median_house_value", predictionCol="prediction_numeric", metricName="mae") 
evaluator_r2_numeric = RegressionEvaluator(labelCol="median_house_value", predictionCol="prediction_numeric", metricName="r2")

rmse_numeric = evaluator_rmse_numeric.evaluate(predictions_numeric)
mae_numeric = evaluator_mae_numeric.evaluate(predictions_numeric)
r2_numeric = evaluator_r2_numeric.evaluate(predictions_numeric)

print(" МОДЕЛЬ 2 (ТОЛЬКО ЧИСЛОВЫЕ ФИЧИ):")
print(f"   RMSE: {rmse_numeric:.2f}")
print(f"   MAE:  {mae_numeric:.2f}")
print(f"   R²:   {r2_numeric:.4f}")

print("\n" + "="*50)
print("СРАВНЕНИЕ МОДЕЛЕЙ:")
print("="*50)
print(f"{'Метрика':<10} {'Модель 1':<12} {'Модель 2':<12} {'Разница':<10}")
print(f"{'-'*50}")
print(f"{'RMSE':<10} {rmse_all:<12.2f} {rmse_numeric:<12.2f} {rmse_all-rmse_numeric:<10.2f}")
print(f"{'MAE':<10} {mae_all:<12.2f} {mae_numeric:<12.2f} {mae_all-mae_numeric:<10.2f}")
print(f"{'R²':<10} {r2_all:<12.4f} {r2_numeric:<12.4f} {r2_all-r2_numeric:<10.4f}")

 МОДЕЛЬ 2 (ТОЛЬКО ЧИСЛОВЫЕ ФИЧИ):
   RMSE: 71791.60
   MAE:  51804.75
   R²:   0.6275

СРАВНЕНИЕ МОДЕЛЕЙ:
Метрика    Модель 1     Модель 2     Разница   
--------------------------------------------------
RMSE       70786.68     71791.60     -1004.91  
MAE        50863.76     51804.75     -940.99   
R²         0.6378       0.6275       0.0104    


**Выводы:**

Модель 1 (со всеми фичами) показала лучшее качество по всем метрикам
Категориальная переменная ocean_proximity улучшает предсказания - добавление OHE дало прирост в R²
RMSE и MAE показывают, что средняя ошибка предсказания составляет около $50,000-$70,000
R² = 0.6378 означает, что модель объясняет около 64% дисперсии цен на жилье

**Все требования проекта выполнены:**

- Инициализирована Spark сессия
- Данные загружены и проанализированы
- Пропуски заполнены (медианой)
- Применен One-Hot Encoding для категориальной переменной
- Построены 2 модели линейной регрессии
- Модели оценены по RMSE, MAE, R2
- Проведено сравнение и сделаны выводы